# Chapter 7. Graph Attention Networks 발제 노트


## 이번 장의 핵심
- GAT는 이웃을 모두 같은 비중으로 합치지 않고, 각 이웃의 중요도를 attention으로 계산한다.
- 따라서 같은 그래프 구조라도 어떤 이웃이 더 중요한지 데이터로부터 학습할 수 있다.
- 이 노트북은 먼저 작은 예제로 attention 계산을 손으로 따라가고, 이후 실제 citation graph에서 GAT를 학습시킨다.


### 실행 환경
이 노트북은 현재 수업 저장소의 `.venv`와 Python 3.10 기준으로 실행한다. 노트북 안에서 추가 설치를 반복하지 않고, 환경이 이미 준비되었다는 전제로 진행한다.


In [ ]:
import torch

torch.manual_seed(1)
torch.cuda.manual_seed(1)
torch.cuda.manual_seed_all(1)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


## 1. 작은 행렬 예제로 GAT 계산 따라가기
먼저 4개 노드로 이루어진 작은 그래프를 두고 attention score가 어떻게 만들어지는지 단계별로 확인한다. 이 부분의 목적은 PyG API 사용법이 아니라, GAT의 계산 논리를 눈으로 확인하는 것이다.


In [ ]:
import numpy as np
np.random.seed(0)

A = np.array([
    [1, 1, 1, 1],
    [1, 1, 0, 0],
    [1, 0, 1, 1],
    [1, 0, 1, 1]
])
A

In [ ]:
X = np.random.uniform(-1, 1, (4, 4))
X

In [ ]:
W = np.random.uniform(-1, 1, (2, 4))
W

In [ ]:
W_att = np.random.uniform(-1, 1, (1, 4))
W_att

### 입력 정의
- `A`: 인접행렬
- `X`: 노드 특성 행렬
- `W`: 노드 특성을 새로운 표현으로 바꾸는 선형변환
- `W_att`: 두 노드 쌍의 중요도를 평가하는 attention 파라미터


In [ ]:
connections = np.where(A > 0)
connections

### 이웃 쌍 표현 만들기
각 연결 `(i, j)`에 대해 먼저 노드 `i`와 `j`의 선형변환 결과를 이어 붙인다. GAT는 이 이어 붙인 쌍 표현을 바탕으로 두 노드 사이의 상대적 중요도를 계산한다.


In [ ]:
np.concatenate([(X @ W.T)[connections[0]], (X @ W.T)[connections[1]]], axis=1)

### 비정규화 attention score
이제 `W_att`를 곱해 비정규화 점수 `a_{ij}`를 만든다. 이 값은 아직 확률이 아니라서, 이 단계에서는 이웃마다 중요도의 원시 score만 계산된 상태다.


In [ ]:
a = W_att @ np.concatenate([(X @ W.T)[connections[0]], (X @ W.T)[connections[1]]], axis=1).T
a

### 활성화와 마스킹
GAT는 비정규화 score에 LeakyReLU를 적용해 음수 영역의 기울기를 일부 살린다. 그리고 실제로 연결된 이웃에 대해서만 점수를 유지하고, 없는 간선은 집계에서 제외한다.


In [ ]:
def leaky_relu(x, alpha=0.2):
    return np.maximum(alpha*x, x)

e = leaky_relu(a)
e

In [ ]:
E = np.zeros(A.shape)
E[connections[0], connections[1]] = e[0]
E

### softmax로 attention coefficient 얻기
이제 각 노드가 바라보는 이웃 집합 안에서 softmax를 적용해 가중치 합이 1이 되도록 만든다. 이렇게 얻은 값이 논문에서의 attention coefficient `alpha_{ij}`에 해당한다.


In [ ]:
def softmax2D(x, axis):
    e = np.exp(x - np.expand_dims(np.max(x, axis=axis), axis))
    sum = np.expand_dims(np.sum(e, axis=axis), axis)
    return e / sum

W_alpha = softmax2D(E, 1)
W_alpha

### 최종 집계
정규화된 attention coefficient를 이용해 이웃 표현을 가중합하면 새로운 노드 표현 `H`를 얻는다. 핵심은 모든 이웃을 균등 평균하는 것이 아니라, 더 중요한 이웃에 더 큰 가중치를 준다는 점이다.


In [ ]:
H = A.T @ W_alpha @ X @ W.T
H

## 2. 실제 그래프 데이터에서 GAT 학습
이제 toy example에서 확인한 계산을 PyTorch Geometric의 `GATv2Conv`로 옮긴다. 먼저 `Planetoid` 데이터셋을 불러오고, 이어서 GAT 모델을 정의해 Cora와 CiteSeer에서 실험한다.


In [ ]:
from torch_geometric.datasets import Planetoid

# Import dataset from PyTorch Geometric
dataset = Planetoid(root=".", name="Cora")
data = dataset[0]

### 모델 구조 해설
이 구현은 2-layer GAT다.
- 첫 번째 레이어는 multi-head attention으로 표현력을 높인다.
- 두 번째 레이어는 클래스로 바로 투영한다.
- dropout과 ELU를 함께 써서 과적합과 학습 불안정을 줄인다.


In [ ]:
import torch
torch.manual_seed(1)
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, GCNConv
from torch.nn import Linear, Dropout


def accuracy(y_pred, y_true):
    """Calculate accuracy."""
    return torch.sum(y_pred == y_true) / len(y_true)


class GAT(torch.nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, heads=8):
        super().__init__()
        self.gat1 = GATv2Conv(dim_in, dim_h, heads=heads)
        self.gat2 = GATv2Conv(dim_h*heads, dim_out, heads=1)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=0.6, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.gat2(h, edge_index)
        return F.log_softmax(h, dim=1)

    def fit(self, data, epochs):
        criterion = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=0.01)

        self.train()
        for epoch in range(epochs+1):
            optimizer.zero_grad()
            out = self(data.x, data.edge_index)
            loss = criterion(out[data.train_mask], data.y[data.train_mask])
            acc = accuracy(out[data.train_mask].argmax(dim=1), data.y[data.train_mask])
            loss.backward()
            optimizer.step()

            if(epoch % 20 == 0):
                val_loss = criterion(out[data.val_mask], data.y[data.val_mask])
                val_acc = accuracy(out[data.val_mask].argmax(dim=1), data.y[data.val_mask])
                print(f'Epoch {epoch:>3} | Train Loss: {loss:.3f} | Train Acc: {acc*100:>5.2f}% | Val Loss: {val_loss:.2f} | Val Acc: {val_acc*100:.2f}%')

    @torch.no_grad()
    def test(self, data):
        self.eval()
        out = self(data.x, data.edge_index)
        acc = accuracy(out.argmax(dim=1)[data.test_mask], data.y[data.test_mask])
        return acc

# Create the Vanilla GNN model
gat = GAT(dataset.num_features, 32, dataset.num_classes)
print(gat)

# Train
gat.fit(data, epochs=100)

# Test
acc = gat.test(data)
print(f'GAT test accuracy: {acc*100:.2f}%')

### 데이터셋 전환 포인트
앞에서는 `Cora`로 모델 구조를 확인했고, 여기서는 `CiteSeer`로 바꿔 다시 학습한다. 두 citation network는 차수 분포와 분류 난이도가 달라서, attention의 장점이 어디에서 드러나는지 비교하기 좋다.


In [ ]:
dataset = Planetoid(root=".", name="CiteSeer")
data = dataset[0]
data

### 차수 분포를 왜 보나
노드 차수는 한 노드가 몇 개의 이웃과 연결되어 있는지를 뜻한다. GAT는 모든 이웃을 동일하게 보지 않기 때문에, 차수가 높은 노드와 낮은 노드에서 성능이 다르게 나타날 수 있다.


In [ ]:
import matplotlib.pyplot as plt
from torch_geometric.utils import degree
from collections import Counter

# Get list of degrees for each node
degrees = degree(dataset[0].edge_index[0]).numpy()

# Count the number of nodes for each degree
numbers = Counter(degrees)

# Bar plot
fig, ax = plt.subplots()
ax.set_xlabel('Node degree')
ax.set_ylabel('Number of nodes')
plt.bar(numbers.keys(), numbers.values())

### CiteSeer에서 다시 학습
이제 같은 GAT 구조를 CiteSeer에 적용한다. 발표에서는 정확도 숫자 자체보다, 모델이 degree imbalance와 sparse neighborhood 상황에서 어떤 장점을 보일 수 있는지를 함께 설명하면 좋다.


In [ ]:
# Create the Vanilla GNN model
gat = GAT(dataset.num_features, 16, dataset.num_classes)
print(gat)

# Train
gat.fit(data, epochs=100)

# Test
acc = gat.test(data)
print(f'GAT test accuracy: {acc*100:.2f}%')

### 결과 해석 포인트
마지막 셀은 노드 차수별 정확도를 비교한다.
- 차수가 낮은 노드는 참고할 이웃 정보가 적다.
- 차수가 높은 노드는 정보가 많지만, 모든 이웃이 유용한 것은 아니다.
- GAT는 이런 상황에서 중요한 이웃에 집중함으로써 단순 평균 방식보다 유리할 수 있다.
발표에서는 막대그래프 숫자 자체보다, 왜 차수에 따라 성능 차이가 생기는지 설명하는 쪽이 더 중요하다.


In [ ]:
# Get model's classifications
out = gat(data.x, data.edge_index)

# Calculate the degree of each node
degrees = degree(data.edge_index[0]).numpy()

# Store accuracy scores and sample sizes
accuracies = []
sizes = []

# Accuracy for degrees between 0 and 5
for i in range(0, 6):
    mask = np.where(degrees == i)[0]
    accuracies.append(accuracy(out.argmax(dim=1)[mask], data.y[mask]))
    sizes.append(len(mask))

# Accuracy for degrees > 5
mask = np.where(degrees > 5)[0]
accuracies.append(accuracy(out.argmax(dim=1)[mask], data.y[mask]))
sizes.append(len(mask))

# Bar plot
fig, ax = plt.subplots()
ax.set_xlabel('Node degree')
ax.set_ylabel('Accuracy score')
plt.bar(['0','1','2','3','4','5','6+'], accuracies)
for i in range(0, 7):
    plt.text(i, accuracies[i], f'{accuracies[i]*100:.2f}%', ha='center', color='black')
for i in range(0, 7):
    plt.text(i, accuracies[i]//2, sizes[i], ha='center', color='white')